# 03 — Close thesis gaps

1. Confirm GoEmotions metrics  
2. Finish **CREMA-D LSTM** (biggest gap vs report)  
3. Finish **BanglaSER BiLSTM**  
4. Smoke-test fusion (WAS/CED)  

**Runtime → T4 GPU.** Upload updated `src/` to Drive first.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

os.environ["HF_HOME"] = "/content/hf_cache"
os.chdir("/content")
from google.colab import drive
drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive/audio-sentiment-analysis")
WORK = Path("/content/audio-sentiment-analysis")
WORK.mkdir(exist_ok=True)

# refresh code only
for name in ["requirements.txt"]:
    shutil.copy2(DRIVE / name, WORK / name)
for name in ["src", "configs"]:
    if (WORK / name).exists():
        shutil.rmtree(WORK / name)
    shutil.copytree(DRIVE / name, WORK / name, ignore=shutil.ignore_patterns("__pycache__", "*.pyc"))

os.chdir(WORK)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
import torch
print("device:", "cuda" if torch.cuda.is_available() else "cpu")
assert torch.cuda.is_available(), "Enable T4 GPU"

## A) GoEmotions / text metrics
Reads `models_final/text/**/*metrics.json` on Drive.

In [ ]:
!python -m src.text.print_metrics

## B) Restore audio caches from Drive (if present)
Copies `models_final/audio` → `models/audio` so LLD caches / checkpoints resume.

In [ ]:
from pathlib import Path
import shutil

src = Path("/content/drive/MyDrive/audio-sentiment-analysis/models_final/audio")
dst = Path("/content/audio-sentiment-analysis/models/audio")
dst.mkdir(parents=True, exist_ok=True)
if src.exists():
    for p in src.rglob("*"):
        if p.is_file():
            t = dst / p.relative_to(src)
            t.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(p, t)
    print("restored from models_final/audio")
else:
    print("no models_final/audio yet")

for ds in ["crema_d", "banglaser"]:
    feat = dst / ds / "dnn" / "features.npz"
    best = dst / ds / "dnn" / "best.pt"
    print(ds, "features.npz", feat.exists(), "best.pt", best.exists())

## C) Ensure wavs exist under /content
Skip download if already present from earlier session.

In [ ]:
from pathlib import Path
import shutil

crema = Path("/content/audio-sentiment-analysis/data/audio/crema_d")
bangla = Path("/content/audio-sentiment-analysis/data/audio/banglaser")
print("crema", len(list(crema.rglob("*.wav"))) if crema.exists() else 0)
print("bangla", len(list(bangla.rglob("*.wav"))) if bangla.exists() else 0)

# CREMA-D quick restore if empty
if not crema.exists() or len(list(crema.rglob("*.wav"))) < 1000:
    !apt-get -qq install -y git-lfs
    !git lfs install
    %cd /content
    !rm -rf CREMA-D
    !GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 https://github.com/CheyneyComputerScience/CREMA-D.git
    %cd /content/CREMA-D
    !git lfs pull --include="AudioWAV/*"
    crema.mkdir(parents=True, exist_ok=True)
    !cp /content/CREMA-D/AudioWAV/*.wav {crema}/
    %cd /content/audio-sentiment-analysis
    print("crema now", len(list(crema.rglob("*.wav"))))

# BanglaSER from Drive zip if empty
if not bangla.exists() or len(list(bangla.rglob("*.wav"))) < 100:
    z = Path("/content/drive/MyDrive/BanglaSER.zip")
    assert z.exists(), "Upload BanglaSER.zip to MyDrive"
    import zipfile
    raw = Path("/content/BanglaSER_raw")
    if raw.exists(): shutil.rmtree(raw)
    raw.mkdir()
    bangla.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(z) as zf:
        zf.extractall(raw)
    for wav in raw.rglob("*.wav"):
        shutil.copy2(wav, bangla / wav.name)
    print("bangla now", len(list(bangla.rglob("*.wav"))))

## D) Finish CREMA-D LSTM (thesis ~79–83%)
Resumes `features.npz` / `best.pt` if present. Class-weighted CE enabled.

In [ ]:
%cd /content/audio-sentiment-analysis
!python -m src.audio.train_dnn --dataset crema_d --epochs 20 --batch-size 32

## E) Finish BanglaSER BiLSTM (must beat XGB ~0.80)
Only swap XGB in the final system if BiLSTM wins.

In [ ]:
%cd /content/audio-sentiment-analysis
!python -m src.audio.train_dnn --dataset banglaser --epochs 20 --batch-size 16

## F) Fusion smoke test + copy finals to Drive

In [ ]:
%cd /content/audio-sentiment-analysis
!python -m src.fusion.demo

from pathlib import Path
import shutil
for ds in ["crema_d", "banglaser"]:
    p = Path(f"models/audio/{ds}/dnn/dnn_metrics.json")
    print(ds, p.read_text() if p.exists() else "MISSING")

DRIVE_FINAL = Path("/content/drive/MyDrive/audio-sentiment-analysis/models_final/audio")
if DRIVE_FINAL.exists():
    shutil.rmtree(DRIVE_FINAL)
copied = 0
for p in Path("models/audio").rglob("*"):
    if not p.is_file() or "runs" in p.parts:
        continue
    dest = DRIVE_FINAL / p.relative_to("models/audio")
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(p, dest)
    copied += 1
print("copied", copied, "->", DRIVE_FINAL)
!du -sh "{DRIVE_FINAL}"